# JacobianFactorQ

A `JacobianFactorQ` is one of the specialized Jacobian factor classes used internally by smart factors. It represents the orthogonal null-space projection produced when a smart projection factor eliminates its landmark.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/slam/doc/JacobianFactorQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [2]:
import gtsam
import numpy as np

from gtsam.symbol_shorthand import X

## Obtain one from a smart factor

A `JacobianFactorQ` is normally returned by smart-factor linearization rather than constructed directly. We create two calibrated cameras, add their measurements of one landmark to a smart projection factor, and call `createJacobianQFactor()` to eliminate the landmark. Because each camera has an 11-dimensional tangent vector, this example returns `JacobianFactorQ112`.

In [3]:
calibration = gtsam.Cal3_S2(500.0, 500.0, 0.0, 320.0, 240.0)
cameras = [
    gtsam.PinholeCameraCal3_S2(gtsam.Pose3(), calibration),
    gtsam.PinholeCameraCal3_S2(
        gtsam.Pose3(gtsam.Rot3(), np.array([1.0, 0.0, 0.0])),
        calibration,
    ),
]
landmark = np.array([0.0, 0.0, 5.0])
factor = gtsam.SmartProjectionFactorPinholeCameraCal3_S2(
    gtsam.noiseModel.Isotropic.Sigma(2, 1.0)
)
values = gtsam.Values()
for index, camera in enumerate(cameras):
    factor.add(camera.project(landmark), X(index))
    values.insert(X(index), camera)

jacobian = factor.createJacobianQFactor(values, 0.0)
assert isinstance(jacobian, gtsam.JacobianFactorQ112)
print("type:", type(jacobian).__name__)
print("keys:", jacobian.keys())
print("A shape:", jacobian.getA().shape)
print("b shape:", jacobian.getb().shape)

type: JacobianFactorQ112
keys: [8646911284551352320, 8646911284551352321]
A shape: (4, 22)
b shape: (4,)


## Inspect the projected linear system

The returned object behaves like any other `JacobianFactor`. `keys()` identifies the camera variables, while `getA()` and `getb()` expose the whitened linear system. The dimension checks below show how the two camera blocks appear in the matrix.

In [4]:
assert jacobian.rows() == jacobian.getA().shape[0]
assert jacobian.getA().shape[1] == 11 * len(cameras)
assert jacobian.getb().shape == (jacobian.rows(),)

## When to use it

`JacobianFactorQ` is an internal linear factor for smart factors. Most applications select the Jacobian-Q mode through [`SmartFactorParams`](SmartFactorParams.ipynb) and pass the returned factor to a Gaussian solver rather than manipulating it directly. [`SmartProjectionFactor`](SmartProjectionFactor.ipynb) shows the higher-level nonlinear factor.

## Source

[`JacobianFactorQ.h`](../JacobianFactorQ.h)


## AI assistance caveat

AI was used to help draft this documentation, and inaccuracies could be present.